# Giọng gomastergi (finetune r16) trên Colab
Model đã up sẵn trên Drive của ông: `gomastergi-r16.zip` (thư mục gốc My Drive).
Chạy lần lượt từng cell bên dưới. Colab có GPU T4 miễn phí nên đọc rất nhanh.

In [ ]:
# 1. Mount Drive + giải nén model + cài thư viện
from google.colab import drive
drive.mount('/content/drive')

!unzip -o "/content/drive/MyDrive/gomastergi-r16.zip" -d /content/
!pip install -q vieneu transformers


In [ ]:
# 2. Tải model + đọc thử 1 câu
from vieneu import Vieneu
from IPython.display import Audio

tts = Vieneu(mode="v3turbo", backbone_repo="/content/final-gomastergi-r16")
audio = tts.infer("Xin chào các bạn, hôm nay mình sẽ hướng dẫn các bạn cách build đội hình cho người mới chơi Genshin Impact nhé.", voice="gomastergi")
tts.save(audio, "/content/output.wav")
Audio("/content/output.wav")

In [ ]:
# 3. Đọc câu của ông: sửa text rồi chạy lại cell này
from IPython.display import Audio

text = "Nhập câu cần đọc vào đây nhé."
audio = tts.infer(text, voice="gomastergi")
tts.save(audio, "/content/output.wav")
Audio("/content/output.wav")

In [ ]:
# 4. (Tùy chọn) Mở giao diện web với giọng gomastergi trong menu
# Chỉ chạy cell này nếu ông thích dùng giao diện web thay cho code.
# Lưu ý: nút Custom Model trong web UI KHÔNG load được model v3 (nó dùng backend cũ),
# nên cell này chèn thẳng giọng gomastergi vào menu chính của web UI.
# Repo đang để riêng tư nên cần token HF 1 lần — dán token vào dòng dưới.
import os
os.environ["HF_TOKEN"] = "hf_..."  # <-- dán token HF của ông vào đây

%cd /content
!rm -rf VieNeu-TTS
!git clone -q https://github.com/pnnbao97/VieNeu-TTS.git
p = "VieNeu-TTS/apps/gradio_main.py"
s = open(p, encoding="utf-8").read()
entry = '''
filtered_backbones["VieNeu-TTS-v3-Turbo-gomastergi"] = {
    "repo": "ntrung/VieNeu-TTS-v3-Turbo-gomastergi",
    "precision": "fp32",
    "supports_streaming": False,
    "description": "Giong gomastergi finetune (rieng tu).",
}
'''
anchor = 'if not HAS_GPU:'
assert anchor in s and 'gomastergi' not in s, "da patch roi hoac khong tim thay cho chen"
open(p, "w", encoding="utf-8").write(s.replace(anchor, entry + anchor, 1))
print("Đã thêm giọng vào menu web UI. Mở web rồi chọn 'VieNeu-TTS-v3-Turbo-gomastergi' và bấm Tải Model.")
%cd VieNeu-TTS
!uv sync --extra cuda -q
!uv run vieneu-web
